# GPT-2 stack smoke test

Covers UCSC's LLM course: tiktoken with GPT-2's split pattern, GPT-2
weights loaded through safetensors, MMLU-style parquet read with pyarrow,
and a transformers GPT-2 that generates.

Everything is built locally so the test never touches the network.

In [ ]:
import pyarrow
import regex
import safetensors
import tiktoken
import tokenizers
import torch
import transformers

for m in (torch, transformers, tokenizers, tiktoken, safetensors, pyarrow, regex):
    print(m.__name__, m.__version__)

In [ ]:
# tiktoken with GPT-2's pre-tokenizer pattern. Byte-level ranks only, since
# the real gpt2 encoding is downloaded on first use.
GPT2_PAT = r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
enc = tiktoken.Encoding(
    name="gpt2-bytes",
    pat_str=GPT2_PAT,
    mergeable_ranks={bytes([i]): i for i in range(256)},
    special_tokens={},
)
text = "Hello world, it's 2026."
ids = enc.encode(text)
assert enc.decode(ids) == text
print(regex.findall(GPT2_PAT, text))

In [ ]:
# A tiny GPT-2, saved and reloaded through safetensors -- the course loads
# the released weights this way.
import os, tempfile
from safetensors.torch import load_file, save_file
from transformers import GPT2Config, GPT2LMHeadModel

config = GPT2Config(n_layer=2, n_head=2, n_embd=64, vocab_size=256, n_positions=64)
model = GPT2LMHeadModel(config).eval()

path = os.path.join(tempfile.mkdtemp(), "model.safetensors")
state = {k: v.contiguous() for k, v in model.state_dict().items() if k != "lm_head.weight"}
save_file(state, path)

reloaded = GPT2LMHeadModel(config).eval()
missing, unexpected = reloaded.load_state_dict(load_file(path), strict=False)
assert not unexpected and set(missing) <= {"lm_head.weight"}
reloaded.tie_weights()
print(f"{sum(p.numel() for p in reloaded.parameters()):,} parameters, "
      f"{len(reloaded.transformer.h)} blocks")

In [ ]:
# Short generation on the reloaded weights.
prompt = torch.tensor([enc.encode("Hi")])
with torch.no_grad():
    out = reloaded.generate(prompt, max_new_tokens=8, do_sample=False,
                            pad_token_id=config.vocab_size - 1)
assert out.shape[1] == prompt.shape[1] + 8
print(tuple(prompt.shape), "->", tuple(out.shape))

In [ ]:
# MMLU ships as parquet.
import pyarrow as pa
import pyarrow.parquet as pq

table = pa.table({"question": ["2+2?"], "choices": [["3", "4"]], "answer": [1]})
pq_path = os.path.join(tempfile.mkdtemp(), "mmlu.parquet")
pq.write_table(table, pq_path)
df = pq.read_table(pq_path).to_pandas()
assert df.loc[0, "answer"] == 1
df